# Exact reproduction: Budget-Aware V10

This notebook reproduces the four-file `submission.zip` from
[flexonafft V10](https://www.kaggle.com/code/flexonafft/gemma-4-budget-aware-coding-agent-v2?scriptVersionId=357955943).
It was the highest-scoring public Code notebook when checked on 24 October 2026,
with a linked public score of **0.22**. That score belongs to the upstream submission;
our reproduction needs its own competition evaluation.

The prompt, model, nine tools, sampling and evaluation configuration are unchanged.
The final cell requires a byte-for-byte identical ZIP: 3,713 bytes, SHA-256
`6ff73d104725cc31deadd18787f3f55e5ce534889cec82387e3b280637d34887`.
No V1 prompt edits or additional archive members are included.

Run all cells to build `/kaggle/working/submission.zip`. Packaging needs no GPU,
model download or network access. Gemma inference happens during competition evaluation.
The repository also retains our separate V1 experiment, whose public score was 0.15.

Original attribution: Igor Zharov / flexonafft; based on matterhorn3838's
[Gemma 4 SuperAgent](https://www.kaggle.com/code/matterhorn3838/gemma-4-superagent),
which credits the official starter and community graph-tool investigations.
Apache-2.0 licensing and credits are preserved in the repository LICENSE and NOTICE.txt.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, os, tempfile, zipfile
import yaml

OUTPUT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd() / 'local_output'
OUTPUT.mkdir(parents=True, exist_ok=True)
AGENT_DIR = OUTPUT / 'agent_bundle'
AGENT_DIR.mkdir(exist_ok=True)
print('Output:', OUTPUT)


In [ ]:
# Generated from the repository source; regenerate after editing.
CANDIDATE_ID = 'budget-aware-v10-exact-reproduction'
FILES = {'agent.yaml': 'name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 80\n', 'prompts/system.md': 'You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.\n\n# HARD LIMITS\n- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.\n- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`). This is an automatic failure.\n- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.\n- Never look outside /workspace. All dependencies are already installed.\n- Never use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`. They return nothing useful.\n\n# CALL PLAN\n- Calls 1-8: locate and read. Calls 9-12: edit. You MUST have edited a source file by call 12. If you are unsure, make your best-guess edit in the most likely place now. A wrong edit can still be corrected. Pure reading cannot score.\n- Calls 13-20: verify and correct.\n- Call 20 onward: no new exploration. Check, fix, submit.\n- Always finish with `git diff` followed by `submit_patch`.\n\n# PROCEDURE\n\n## 1. Extract the specification (no tool call)\nFrom the issue, list for yourself: the exact file, function, class or CLI name; the exact error message and exception type; the expected return value or output; the inputs that trigger the bug; every edge case mentioned (None, empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue as a literal that must appear exactly in your code.\n\n## 2. Locate\n- Use exact paths from the issue directly. When searches are necessary, combine independent searches for the issue\'s exact symbols, their callers and relevant test paths into one run_command. Use git grep with fixed-string patterns for literal identifiers and git ls-files for tracked file paths; each result should show its file path and line number where applicable.\n- Batch only searches whose inputs are already known. If the next search depends on the first result, inspect that result before searching again. Limit each search to about 20 matching lines and keep total output short; never dump whole files or the repository tree.\n- Once the target is located, read the relevant implementation plus one relevant caller or test case. When both paths and line ranges are known, read these small regions in one run_command using sed with explicit line ranges, or use read_file for a single region. Aim for about 40 lines per region and at most three regions per command. Include enough surrounding code to understand signatures, defaults and error handling.\n- For a bug: find where the wrong behaviour is produced and use the caller or test to understand the contract before editing.\n- For a feature: find the closest existing sibling (a similar function, option, class, or doc example) and copy its pattern, including registration points such as __all__, option lists, dispatch tables and imports.\n- Reuse paths and test names already discovered. Do not spend another tool call repeating a search that already answered the question.\n- If the issue includes a code snippet that reproduces the problem, you may run it once with python -c (one call, never create a file) to see the real error.\n\n## 3. Edit\n- Use edit_file for changes to existing code. Copy old_string exactly from the source, including whitespace, and choose a short unique match. Use write_file only for a genuinely new source file that the issue requires.\n- If edit_file reports no match or multiple matches, read the exact source region and retry with a smaller unique snippet. Never repeat an identical failed edit. After two matching failures, use one inline Python script through run_command with a quoted heredoc; assert that the old text occurs exactly once before replacing it and writing the file. Preserve unrelated content and do not rewrite the file from memory.\n- Fix the cause, not the symptom. Do not special-case the example from the issue.\n- Change only what the issue requires. Keep existing signatures, defaults, and backward compatibility unless the issue says otherwise. Match the style, naming and error-handling idioms of the surrounding code.\n- Use the exact exception class and exact message text from the issue. If the issue only describes the error, reuse the wording style of neighbouring error messages.\n- Handle every edge case you listed in step 1.\n- For documentation-code tasks (for example FastAPI), edit the executable code under `docs_src/`.\n\n## 4. Verify (at most 2 calls)\n- Find the test file with `find tests -name "*keyword*.py"`, then run `timeout 150 python -m pytest tests/path/test_x.py -x -q -k keyword`. Use `python3 -m unittest tests.test_x` for unittest projects.\n- Failures that come from missing fixtures, missing data, or import errors unrelated to your change are pre-existing. Ignore them. Never repair tests or the environment.\n- If a test related to your change fails, read the traceback, fix the source once, and rerun once. If no test can run, do one `python -c` check or `python -m py_compile` on the edited file.\n\n## 5. Final check and submit\n- Run `git diff`. Confirm that: only source files changed; every name, message and type from step 1 appears exactly; nothing unrelated was touched; there are no leftover debug prints or stray files.\n- Call `submit_patch`. Confirm `patch_size > 0` and `files_changed > 0`. If either is 0, you have not edited anything. Edit now and submit again.\n- End with a single-sentence summary of the fix.\n\n# DECISION RULES\n- Ambiguous issue: choose the most literal reading of the issue text and act on it. Do not ask questions and do not stall.\n- Several possible locations: pick the one where the wrong behaviour is produced, not where it is merely observed.\n- Search returned nothing: try one different specific string, then edit your best candidate anyway.\n- Stuck after a failed fix: make a different, smaller change. Do not repeat the same edit.\n- Never conclude that nothing needs to change. Every task requires a non-empty source patch.\n'}
EXPECTED_MEMBER_HASHES = {'agent.yaml': '680799291cc7d156d06fb7bfb5577f683ccc0d555b00e8ddb662d577f9bfa5e4', 'configs/sampling.yaml': '571a934e89911734927533f1d36cbcc7b8d493592febe63b217e911cf7ed4926', 'eval_config.yaml': '916cf6cb34a6cc3de39837fa72cac681599b9ac2a3ee0b893adadfd36474c531', 'prompts/system.md': '7f8dde62578a247ffbfc1a519b712ae65849fcfae1ee70c49e0628cab74cbef6'}


In [ ]:
ALLOWED = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
assert set(FILES) == set(EXPECTED_MEMBER_HASHES)
assert 'agent.yaml' in FILES
for name, content in FILES.items():
    relative = PurePosixPath(name)
    assert not relative.is_absolute() and '..' not in relative.parts
    assert relative.suffix.lower() in ALLOWED
    assert hashlib.sha256(content.encode('utf-8')).hexdigest() == EXPECTED_MEMBER_HASHES[name]
    path = AGENT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(content.encode('utf-8'))

class Loader(yaml.SafeLoader):
    pass
def include(loader, node):
    name = loader.construct_scalar(node)
    assert name in FILES, name
    return FILES[name] if name.endswith(('.md', '.txt')) else yaml.safe_load(FILES[name])
Loader.add_constructor('!include', include)
agent = yaml.load(FILES['agent.yaml'], Loader=Loader)
generation = agent['generate_content_config']
evaluation = yaml.safe_load(FILES['eval_config.yaml'])['evaluation']
assert agent['model'] in {'gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-12b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct', 'gemma-4-e2b-it-qat-w4a16-ct'}
assert 'submit_patch' in agent['tools']
assert 1 <= generation['max_output_tokens'] <= 32768
assert 0 <= generation.get('thinking_config', {}).get('thinking_budget', 4096) <= 32768
assert all(evaluation[key] > 0 for key in ('timeout_seconds', 'max_tool_calls', 'max_time_minutes', 'max_turns'))
assert len(agent['instruction']) < 1000000
assert sum(len(v.encode('utf-8')) for v in FILES.values()) < 3 * 1024**3
print('Archive contract, model, tools, include paths and budgets passed.')


In [ ]:
archive_path = OUTPUT / 'submission.zip'
with zipfile.ZipFile(archive_path, 'w') as archive:
    for name, content in sorted(FILES.items()):
        info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
        info.create_system = 3
        info.external_attr = 0o100644 << 16
        info.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(info, content.encode('utf-8'))
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(FILES)
    for name, content in FILES.items():
        assert archive.read(name) == content.encode('utf-8')
receipt = {
    'candidate': globals().get('CANDIDATE_ID', 'budget-aware-verified-v1'),
    'archive_sha256': hashlib.sha256(archive_path.read_bytes()).hexdigest(),
    'archive_bytes': archive_path.stat().st_size,
    'members': {n: hashlib.sha256(v.encode('utf-8')).hexdigest() for n,v in FILES.items()},
    'packaging_checks_passed': True,
    'leaderboard_score': None,
    'model_quality_measured_by_this_notebook': False,
}
(OUTPUT / 'build_receipt.json').write_text(json.dumps(receipt, indent=2), encoding='utf-8')
print(json.dumps(receipt, indent=2))
print('Ready:', archive_path)


Upload `/kaggle/working/submission.zip` through File Upload or `KaggleApi.competition_submit`. The packaging notebook does not run hidden evaluation. See docs/submission.md for authentication, validation and status checks.

In [ ]:
# Match the scored public artifact, including the ZIP container bytes.
assert receipt['archive_sha256'] == '6ff73d104725cc31deadd18787f3f55e5ce534889cec82387e3b280637d34887'
assert receipt['archive_bytes'] == 3713
receipt['exact_public_artifact_reproduced'] = True
receipt['upstream_url'] = 'https://www.kaggle.com/code/flexonafft/gemma-4-budget-aware-coding-agent-v2?scriptVersionId=357955943'
(OUTPUT / 'build_receipt.json').write_text(json.dumps(receipt, indent=2), encoding='utf-8')
print('Exact public V10 artifact reproduced. Our leaderboard result requires separate evaluation.')
